# Skull acoustic transparency at **500 kHz** — Colab

Time-reversal skull *transparency maps* + transducer placement, end to end, on a real benchmark skull, small enough for a free Colab GPU. A **manuscript-style figure at every step**.

**Pipeline:** `prepare` (skull map → solver inputs) → `sim outward --run` (full-wave GPU solve) → `extract` (→ Field Bundle) → `transparency` / `place`.

**Skull:** the ITRUSST transcranial-ultrasound benchmark skull (Aubry et al., *JASA* 2022), which is defined at 500 kHz. Its surface meshes are rasterized to homogeneous cortical bone (`c=2800`, water `1500`).

**Grid & runtime.** `dx = c0 / (f0·ppw)`, run at **6 PPW (~0.5 mm)** — benchmark-grade. The whole-skull solve writes a decimated full-field dump (`genout_mod.dat`) that `extract` loads into RAM; at 6 PPW over the full padded domain this is **large (tens–hundreds of GB) with the solver's current fixed field decimation** (`modT=8`, `modX=modY=modZ=2`). So the **real whole skull at 6 PPW needs a workstation / High-RAM host**, not free Colab. For a Colab run, either `USE_REAL_SKULL = False` (toy shell) or a cropped region. Reducing the field-dump (coarser `modT`/`modX`) is the lever to make 6 PPW tractable — see the notes below.

## 0 · Runtime

**Runtime ▸ Change runtime type ▸ GPU (T4)**, then run the cells top to bottom.

In [ ]:
# --- setup: install the package + fetch the CUDA solver binary + mesh tooling ---
import os, sys
IN_COLAB = "google.colab" in sys.modules

# skull_transparency (Apache-2.0) + trimesh (rasterize the skull STL).
!pip -q install "git+https://github.com/pinton-lab/skull_transparency.git" trimesh

# The full-wave CUDA solver binary ships in the (public) fullwave2-ultra repo
# (PolyForm Noncommercial -- academic use). Clone it and point FULLWAVE2_BIN at it.
!git clone --depth 1 https://github.com/pinton-lab/fullwave2-ultra.git /content/fullwave2-ultra 2>/dev/null || true
os.environ["FULLWAVE2_BIN"] = "/content/fullwave2-ultra/bin/bench_3d_opt"
print("solver:", os.environ["FULLWAVE2_BIN"], "| exists:", os.path.exists(os.environ["FULLWAVE2_BIN"]))
!nvidia-smi -L || echo "NO GPU visible -- enable a GPU runtime for Section 2."

## Figure helpers (manuscript style)

One reusable set of multi-panel figures, used at each step below. They work on both the no-GPU synthetic bundle (Section 1) and the real solved bundle (Section 2).

In [ ]:
import numpy as np, matplotlib.pyplot as plt

INK, EDGE, FOCUS = "#1d1d1f", "#2c6fbf", "#c0392b"
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 10, "figure.facecolor": "white",
                     "axes.facecolor": "white", "axes.titlesize": 10, "savefig.dpi": 120})

def _ortho(ax3, vol, dx, cmap, vmin, vmax, log=False, titles=("sagittal","coronal","axial")):
    v = np.asarray(vol, float); ctr = np.array(v.shape) // 2
    d = np.log10(np.maximum(v, (v[v>0].min() if (v>0).any() else 1))) if log else v
    lo = np.log10(vmin) if (log and vmin>0) else vmin
    hi = np.log10(vmax) if (log and vmax>0) else vmax
    for a, (s, t) in zip(ax3, zip([d[ctr[0]].T, d[:,ctr[1]].T, d[:,:,ctr[2]].T], titles)):
        im = a.imshow(s, origin="lower", cmap=cmap, vmin=lo, vmax=hi, aspect="equal",
                      extent=[0, s.shape[1]*dx, 0, s.shape[0]*dx])
        a.set_title(t); a.set_xlabel("mm")
    ax3[0].set_ylabel("mm"); return im

def fig_skull(c, dx, title="Skull sound-speed map  c(x)"):
    fig, ax = plt.subplots(1, 3, figsize=(13, 4.3), constrained_layout=True)
    im = _ortho(ax, c, dx, "bone", 1500, float(np.max(c)))
    fig.colorbar(im, ax=ax, fraction=0.02, pad=0.01, label="c (m/s)")
    fig.suptitle(title, fontweight="bold"); plt.show()

def fig_outward(vol, dx, title="Outward field  —  peak intensity (log)"):
    fig, ax = plt.subplots(1, 3, figsize=(13, 4.3), constrained_layout=True)
    v = np.asarray(vol, float); vmax = (v[v>0].max() if (v>0).any() else 1.0)
    im = _ortho(ax, v, dx, "inferno", vmax*1e-4, vmax, log=True)
    fig.colorbar(im, ax=ax, fraction=0.02, pad=0.01, label="log10 intensity")
    fig.suptitle(title, fontweight="bold"); plt.show()

def fig_transparency(t, title="Skull transparency map"):
    fig, ax = plt.subplots(2, 3, figsize=(13, 8.2), constrained_layout=True)
    P = np.asarray(t.surf_vox); val = np.asarray(t.value); vn = val / (val.max() or 1)
    for a, (i, j, ttl) in zip(ax[0], [(1,2,"A-P vs S-I"), (0,2,"R-L vs S-I"), (0,1,"R-L vs A-P")]):
        sc = a.scatter(P[:,i], P[:,j], c=vn, s=6, cmap="inferno", vmin=0, vmax=1, linewidths=0)
        a.set_aspect("equal"); a.set_title(f"transparency  ({ttl})"); a.set_xticks([]); a.set_yticks([])
    fig.colorbar(sc, ax=ax[0], fraction=0.02, pad=0.01, label="normalized transparency")
    ax[1,0].hist(vn, bins=40, color=EDGE, alpha=0.85); ax[1,0].set_title("transparency histogram")
    ax[1,0].set_xlabel("normalized"); ax[1,0].set_ylabel("patches")
    ax[1,1].scatter(np.asarray(t.rad_mm), vn, s=5, color=EDGE, alpha=0.4)
    ax[1,1].set_title("transparency vs source distance"); ax[1,1].set_xlabel("distance (mm)")
    sc2 = ax[1,2].scatter(P[:,0], P[:,2], c=np.asarray(t.Pmax), s=6, cmap="viridis", linewidths=0)
    ax[1,2].set_aspect("equal"); ax[1,2].set_title("peak pressure per patch")
    ax[1,2].set_xticks([]); ax[1,2].set_yticks([]); fig.colorbar(sc2, ax=ax[1,2], fraction=0.046, label="Pmax")
    fig.suptitle(title, fontsize=12, fontweight="bold"); plt.show()

def fig_placement(t, pl, title="Transducer placement"):
    fig, ax = plt.subplots(1, 3, figsize=(14, 4.5), constrained_layout=True,
                           gridspec_kw={"width_ratios": [1, 1, 0.9]})
    P = np.asarray(t.surf_vox); foot = np.zeros(len(P), bool); foot[np.asarray(pl.footprint_surf_idx)] = True
    for a, (i, j, ttl) in zip(ax[:2], [(0,2,"R-L vs S-I"), (1,2,"A-P vs S-I")]):
        a.scatter(P[~foot,i], P[~foot,j], s=5, color="#cccccc", linewidths=0)
        a.scatter(P[foot,i], P[foot,j], s=11, color=FOCUS, linewidths=0, label="bowl footprint")
        a.set_aspect("equal"); a.set_title(f"footprint  ({ttl})"); a.set_xticks([]); a.set_yticks([])
        a.legend(loc="lower right", fontsize=8, frameon=False)
    ax[2].axis("off")
    ax[2].text(0, 0.95, (f"target (MNI mm): {np.round(pl.target_mni_mm,1)}\n"
        f"apex   (MNI mm): {np.round(pl.apex_mni_mm,1)}\nbeam dir: {np.round(pl.beam_dir_mni,2)}\n\n"
        f"incidence:   {pl.incidence_deg:.1f} deg\nfocal length: {pl.focal_length_mm:.0f} mm\n"
        f"footprint:   {pl.n_footprint_patches} patches\ntransparency score: {pl.transparency_score:.3f}"),
        va="top", family="DejaVu Sans Mono", fontsize=10,
        bbox=dict(boxstyle="round,pad=0.5", fc="#f4f6f9", ec=EDGE))
    fig.suptitle(title, fontsize=12, fontweight="bold"); plt.show()

print("figure helpers ready")

## 1 · Quick taste — no GPU, no data

The analysis + figures on a shipped **synthetic** Field Bundle, so you see the whole thing before the GPU solve.

In [ ]:
import skull_transparency as st
b0 = st.load_bundle(st.make_synthetic_bundle("synthetic_bundle"))
t0 = st.compute_transparency_map(b0)
pl0 = st.place_bowl(t0, st.BowlConstraints(focal_length_mm=60.0, bowl_radius_mm=15.0, theta_max_deg=35.0))
print(f"{len(t0.surf_vox)} surface patches | placement incidence {pl0.incidence_deg:.1f} deg, "
      f"score {pl0.transparency_score:.3f}")
fig_transparency(t0, title="Transparency map (synthetic, no GPU)")
fig_placement(t0, pl0, title="Placement (synthetic, no GPU)")

## 2 · Real 500 kHz simulation on the ITRUSST benchmark skull

### 2a · Build the skull sound-speed map  — *figure: input skull*

Downloads the benchmark's inner/outer surface meshes (~13 MB) and rasterizes them to the grid as homogeneous cortical bone at **6 PPW (~0.5 mm)**. The whole real skull at 6 PPW is a workstation / High-RAM job (large field dump); set `USE_REAL_SKULL = False` for the offline toy shell.

In [ ]:
import os, urllib.request, numpy as np

PPW = 6                        # 6 PPW (~0.5 mm) = benchmark-grade. NOTE: the whole-skull field
                               # dump is large at 6 PPW (see the runtime note above) -> needs a
                               # workstation / High-RAM, or USE_REAL_SKULL=False for the toy shell.
USE_REAL_SKULL = True
C0, F0 = 1500.0, 500e3
pitch = round(C0 / (F0 * PPW) * 1e3, 3)     # mm/voxel to match the sim grid

if USE_REAL_SKULL:
    import trimesh
    BASE = ("https://raw.githubusercontent.com/ucl-bug/transcranial-ultrasound-benchmarks"
            "/master/intercomparison/skull-stl")
    for f in ("skull_inner.stl", "skull_outer.stl"):
        if not os.path.exists(f):
            urllib.request.urlretrieve(f"{BASE}/{f}", f)
    outer = trimesh.load("skull_outer.stl"); inner = trimesh.load("skull_inner.stl")
    lo = outer.bounds[0] - 2*pitch
    dims = np.ceil((outer.bounds[1] + 2*pitch - lo) / pitch).astype(int)
    def _raster(mesh):
        vg = mesh.voxelized(pitch=pitch).fill(); idx = np.floor((vg.points - lo) / pitch).astype(int)
        g = np.zeros(dims, bool); ok = np.all((idx >= 0) & (idx < dims), axis=1); idx = idx[ok]
        g[idx[:,0], idx[:,1], idx[:,2]] = True; return g
    bone = _raster(outer) & ~_raster(inner)
    c = np.where(bone, 2800.0, C0).astype(np.float32)
    affine = np.diag([pitch, pitch, pitch, 1.0]); affine[:3, 3] = lo
    print(f"ITRUSST skull @ {pitch} mm: {tuple(int(d) for d in dims)} = {c.size/1e6:.1f} M voxels; "
          f"bone {bone.sum()*pitch**3/1000:.0f} cm^3")
else:
    n = int(round(120 * 0.5 / pitch))
    c = np.full((n, n, n), C0, np.float32)
    zz, yy, xx = np.mgrid[0:n, 0:n, 0:n]
    r = np.sqrt((xx-n/2)**2 + (yy-n/2)**2 + (zz-n/2)**2) * pitch
    c[(r > 40) & (r < 46)] = 2600.0
    affine = np.diag([pitch, pitch, pitch, 1.0]); affine[:3, 3] = -pitch * n / 2
    print("toy shell:", c.shape)

np.save("head_c.npy", c); np.save("affine.npy", affine)
import json; json.dump({"preset": "ctx500", "f0_hz": F0, "ppw": PPW}, open("ctx500.json", "w"))
fig_skull(c, pitch, title=f"Input skull @ {pitch} mm  ({'ITRUSST benchmark' if USE_REAL_SKULL else 'toy shell'})")

### 2b · `prepare` — skull map → solver inputs (no GPU)

Seats an omnidirectional source at the brain center (`--center`) for the whole-skull transparency map, and writes the `.dat` solver deck at `dx = c0/(f0·ppw)`.

In [ ]:
!skull-transparency prepare --c-map head_c.npy --affine affine.npy --center --transducer ctx500.json --out run
import json; m = json.load(open("run/meta.json"))
print(f"sim grid N={m['N']}  dx={m['dX_m']*1e3:.3f} mm  F0={m['F0']/1e3:.0f} kHz")

### 2c · Outward full-wave solve (GPU)  — *figure: outward field*

Runs the CUDA solver; streams the decimated full field to `genout_mod.dat` (which `extract` reads). A few minutes on a GPU at 6 PPW.

In [ ]:
!python -m skull_transparency.sim outward --sim run --out run --run

# fail loudly + actionably if the solve didn't produce the field dump
import os
gm = "run/outward/genout_mod.dat"
assert os.path.exists(gm) and os.path.getsize(gm) > 0, (
    "Solve produced no genout_mod.dat. Check: (1) GPU runtime is ON, (2) the fullwave2-ultra "
    "binary was cloned and $FULLWAVE2_BIN points at it, (3) enough disk (~1 GB at PPW=3).")
print(f"genout_mod.dat: {os.path.getsize(gm)/1e9:.2f} GB")

### 2d · Extract → Field Bundle → transparency map  — *figure: transparency*

In [ ]:
!skull-transparency extract --run run/outward --sim run --out run/bundle

import skull_transparency as st
b = st.load_bundle("run/bundle")
ow = b.outward_iint_pmax(); ow = ow[0] if isinstance(ow, tuple) else ow
fig_outward(np.asarray(ow), b.grid["dx_m"]*1e3, title="Outward field @ 500 kHz  (ITRUSST skull)")

t = st.compute_transparency_map(b)
fig_transparency(t, title="Skull transparency @ 500 kHz  (ITRUSST benchmark skull)")

### 2e · Place a focused window on the map  — *figure: placement*

In [ ]:
pl = st.place_bowl(t, st.BowlConstraints(focal_length_mm=60.0, bowl_radius_mm=32.0, theta_max_deg=35.0))
print(f"incidence {pl.incidence_deg:.1f} deg, transparency score {pl.transparency_score:.3f}")
fig_placement(t, pl, title="Transducer placement on the transparency map")